In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [ ]:
load_dotenv()
openai_client = OpenAI()

In [ ]:
documents = load_faq_data()
index = build_index(documents)

In [ ]:

INSTRUCTIONS = """
You are a question-answering assistant for the LLM Zoomcamp course.

Your task is to answer questions using ONLY the information
provided in the context.

Rules:
1. Treat the context as the only source of truth.
2. Do not use your pretrained knowledge or external knowledge.
3. Do not make assumptions or invent information.
4. If the context does not contain enough information to answer
   the question, respond exactly with:
   I don't know
5. Do not answer questions unrelated to the course.
6. If the answer is available in the context, provide a clear
   and concise answer based on that information.
""".strip()

In [ ]:
assistant = RAGBase(
    index = index,
    instructions=INSTRUCTIONS,
    llm_client=openai_client
)

In [ ]:
answer  = assistant.rag("How to install Ollama locally?")
print(answer)

In [46]:
def search(query):
    boost_dict = {'question': 3.0, 'section' : 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
        filter_dict = filter_dict
    )

In [45]:
search_tool = {
    "type": "function",
    "name":"search",
    "description": 'Search the FAQ database for entries matching the given query',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                'type': 'string',
                'description': 'Seacrh query text to look up in the course FAQ'
            }
        },
        "required":["query"],
        'additionalProperties': False
    }
}

In [16]:
messages = [{'role': 'user', 'content': ' I just discovered the course. Can I join it?'}]
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

call = response.output[0]

In [17]:
import json

In [21]:
args = json.loads(call.arguments)
args

{'query': 'Can I join the course if it has already started? enrolling late discovered course join late'}

In [25]:
results = search(**args)
result_json = json.dumps(results)
print(result_json)

[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}, {"id": "5cc511f85b", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "Does the course certificate show the number of course hours?", "answer": "No. The certificate does not state a total number of hours."}, {"id": "04919992b3", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "How should I start the course and follow the weekly workflow?", "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYo

In [33]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [36]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="I just discovered the course. Can I join it?",
    tools=[search_tool]
)

print(response.output)

call = response.output[0]

result_json = json.dumps(results)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,
    input=[function_call_output],
    tools=[search_tool]
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join after start enrollment access"}', call_id='call_Me0IVJ2ADhiXX0C8t0Sn4Cxj', name='search', type='function_call', id='fc_04366733031eccfd006afe958ce1f887d0839f3bb4b868acb8', async_=None, caller=None, namespace=None, status='completed')]
[ResponseOutputMessage(id='msg_04366733031eccfd006afe958dfb6487d0a2c29d6e4d848424', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [49]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question ='I just discovered the course. Can I join?'

messages = [{
    'role': 'developer', 'content': instructions},
    {'role':'user', 'content': question}
]

In [51]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration"}', call_id='call_ywIVgRR6EmnDIDPekKJB9n0m', name='search', type='function_call', id='fc_0456725eefaef250006afe9907733887d083517c52a862d207', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment can I join after course starts new student"}', call_id='call_FilfAVlgCTbgMfV8OB0HzMxK', name='search', type='function_call', id='fc_0456725eefaef250006afe9907734c87d0b937be9449558392', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join the course registration FAQ"}', call_id='call_FboM8RTgOZvM3cuqhX96p2Mv', name='search', type='function_call', id='fc_0456725eefaef250006afe9907735487d0803ca77438c36c2a', async_=None, caller=None, namespace=None, status='completed')]

In [66]:

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown function: {call.name}")

    # Convert the result into a JSON string
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="I just discovered the course. Can I join it?",
    tools=[search_tool]
)

messages = []

messages.extend(response.output)

for item in response.output:
    print(item)

    if item.type == "function_call":
        function_call_output = make_call(item)
        messages.append(function_call_output)

# Send the function result back to the model
if messages:
    final_response = openai_client.responses.create(
        model="gpt-5.4-mini",
        previous_response_id=response.id,
        input=messages[len(response.output):],
        tools=[search_tool]
    )

    print(final_response.output)


ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after course has started"}', call_id='call_9cVjVYUQW6vMtqBwmjrhLb5P', name='search', type='function_call', id='fc_073b4478b9ea72c6006afe9da00e8087d0962730df25d2450c', async_=None, caller=None, namespace=None, status='completed')
[ResponseOutputMessage(id='msg_073b4478b9ea72c6006afe9da1d23087d094eef830a1ebcbde', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [67]:
messages

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after course has started"}', call_id='call_9cVjVYUQW6vMtqBwmjrhLb5P', name='search', type='function_call', id='fc_073b4478b9ea72c6006afe9da00e8087d0962730df25d2450c', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_9cVjVYUQW6vMtqBwmjrhLb5P',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certificate show the number of course hours?",\n    "answer": "No. The certificate does not state a t